# 청년층 이동 데이터 구조 확인

- 데이터: 월별 청년층 행정동 OD 이동 데이터
- 기간: 1월 ~ 12월
- 목적: 컬럼 구조, 데이터 타입, 결측치, 월별 행 수 확인
- 전처리: 서울 간 이동 외 삭제, 행정동 이름 매핑, 시간 및 날짜 형식 변환
- 활용: 전체 연령 목적 기반 통근 OD와 청년 이동패턴 비교

In [126]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


# 현재 노트북 위치 확인
NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR


# 원본 데이터 폴더
RAW_DIR = PROJECT_ROOT.parent / "project_data" / "raw"

# 전처리 결과 저장 폴더
PROCESSED_DIR = PROJECT_ROOT.parent / "project_data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


print("프로젝트 루트:", PROJECT_ROOT)
print("원본 데이터 폴더:", RAW_DIR)
print("저장 데이터 폴더:", PROCESSED_DIR)

print("원본 폴더 존재:", RAW_DIR.exists())
print("저장 폴더 존재:", PROCESSED_DIR.exists())

프로젝트 루트: /Users/janghwayeong/Desktop/부트캠프_프로젝트/Analysis-of-Youth-Savings-Capacity-Project
원본 데이터 폴더: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/raw
저장 데이터 폴더: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed
원본 폴더 존재: True
저장 폴더 존재: True


In [127]:
def extract_month(file_path: Path) -> int:
    """
    파일명 맨 앞의 월 숫자를 추출한다.

    예:
    1월_이동데이터.csv  -> 1
    12월_이동데이터.csv -> 12
    """
    file_name = file_path.name.strip()
    month_text = ""

    for char in file_name:
        if char.isdigit():
            month_text += char
        else:
            break

    if month_text == "":
        raise ValueError(
            f"파일명 앞에서 월 숫자를 찾지 못했습니다: {file_name}"
        )

    return int(month_text)


csv_files = sorted(RAW_DIR.glob("*.csv"))

monthly_csv_files = []

for file_path in csv_files:
    try:
        month = extract_month(file_path)

        if 1 <= month <= 12:
            monthly_csv_files.append(file_path)

    except ValueError:
        # 숫자로 시작하지 않는 CSV는 제외
        continue


monthly_csv_files = sorted(
    monthly_csv_files,
    key=extract_month
)


print("전체 CSV 파일 수:", len(csv_files))
print("월별 이동 CSV 파일 수:", len(monthly_csv_files))

for file_path in monthly_csv_files:
    print(f"{extract_month(file_path):>2}월 | {file_path.name}")

전체 CSV 파일 수: 13
월별 이동 CSV 파일 수: 12
 1월 | 1월 통합 (0010_cnt, 60plus_cnt, 4050_cnt제거, 출도착시간 필터).csv
 2월 | 2월 통합 (0010_cnt, 60plus_cnt, 4050_cnt제거, 출도착시간 필터).csv
 3월 | 3월 통합 (0010_cnt, 60plus_cnt, 4050_cnt제거, 출도착시간 필터).csv
 4월 | 4월 통합 (0010_cnt, 60plus_cnt, 4050_cnt제거, 출도착시간 필터).csv
 5월 | 5월 통합 (0010_cnt, 60plus_cnt, 4050_cnt제거, 출도착시간 필터).csv
 6월 | 6월 통합 (0010_cnt, 60plus_cnt, 4050_cnt제거, 출도착시간 필터).csv
 7월 | 7월 통합 (0010_cnt, 60plus_cnt, 4050_cnt제거, 출도착시간 필터).csv
 8월 | 8월 통합 (0010_cnt, 60plus_cnt, 4050_cnt제거, 출도착시간 필터).csv
 9월 | 9월 통합 (0010_cnt, 60plus_cnt, 4050_cnt제거, 출도착시간 필터).csv
10월 | 10월 통합 (0010_cnt, 60plus_cnt, 4050_cnt제거, 출도착시간 필터).csv
11월 | 11월 통합 (0010_cnt, 60plus_cnt, 4050_cnt제거, 출도착시간 필터).csv
12월 | 12월 통합 (0010_cnt, 60plus_cnt, 4050_cnt제거, 출도착시간 필터).csv


In [128]:
def read_csv_safely(file_path: Path) -> pd.DataFrame:
    """
    CSV 파일을 여러 인코딩으로 순차적으로 읽는다.
    """
    encodings = [
        "utf-8-sig",
        "utf-8",
        "cp949",
        "euc-kr"
    ]

    last_error = None

    for encoding in encodings:
        try:
            df = pd.read_csv(
                file_path,
                encoding=encoding,
                low_memory=False
            )

            print(
                f"읽기 성공 | {file_path.name} "
                f"| 인코딩: {encoding} "
                f"| 행: {len(df):,}"
            )

            return df

        except UnicodeDecodeError as error:
            last_error = error

    raise ValueError(
        f"파일을 읽지 못했습니다: {file_path.name}\n"
        f"마지막 오류: {last_error}"
    )

In [129]:
required_columns = [
    "o_admdong_cd",
    "d_admdong_cd",
    "st_time_cd",
    "fns_time_cd",
    "move_dist",
    "move_time",
    "2030_cnt",
    "total_cnt",
    "etl_ymd"
]


monthly_data_list = []

for file_path in monthly_csv_files:
    month = extract_month(file_path)

    print(f"\n===== {month}월 처리 시작 =====")

    df = read_csv_safely(file_path)


    # 필요한 컬럼 존재 여부 확인
    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{file_path.name}에 필요한 컬럼이 없습니다: "
            f"{missing_columns}"
        )


    # 이번 전처리에 필요한 컬럼만 유지
    df = df[required_columns].copy()


    # 행정동 코드 정리
    for column in ["o_admdong_cd", "d_admdong_cd"]:
        df[column] = (
            df[column]
            .astype("string")
            .str.strip()
            .str.replace(r"\.0$", "", regex=True)
            .str.zfill(8)
        )


    # 날짜 변환
    df["etl_ymd"] = pd.to_datetime(
        df["etl_ymd"].astype("string"),
        format="%Y%m%d",
        errors="coerce"
    )


    # 숫자형 컬럼 변환
    numeric_columns = [
        "st_time_cd",
        "fns_time_cd",
        "move_dist",
        "move_time",
        "2030_cnt",
        "total_cnt"
    ]

    for column in numeric_columns:
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce"
        )


    # 기준월 추가
    df["기준월"] = month


    # 완전히 동일한 중복 행 제거
    before_duplicate = len(df)

    df = df.drop_duplicates().copy()

    removed_duplicate = before_duplicate - len(df)

    print("중복 제거 행 수:", f"{removed_duplicate:,}")


    monthly_data_list.append(df)

    del df


===== 1월 처리 시작 =====
읽기 성공 | 1월 통합 (0010_cnt, 60plus_cnt, 4050_cnt제거, 출도착시간 필터).csv | 인코딩: utf-8-sig | 행: 24,283,956
중복 제거 행 수: 0

===== 2월 처리 시작 =====
읽기 성공 | 2월 통합 (0010_cnt, 60plus_cnt, 4050_cnt제거, 출도착시간 필터).csv | 인코딩: utf-8-sig | 행: 24,185,418
중복 제거 행 수: 0

===== 3월 처리 시작 =====
읽기 성공 | 3월 통합 (0010_cnt, 60plus_cnt, 4050_cnt제거, 출도착시간 필터).csv | 인코딩: utf-8-sig | 행: 26,030,880
중복 제거 행 수: 0

===== 4월 처리 시작 =====
읽기 성공 | 4월 통합 (0010_cnt, 60plus_cnt, 4050_cnt제거, 출도착시간 필터).csv | 인코딩: utf-8-sig | 행: 27,963,393
중복 제거 행 수: 0

===== 5월 처리 시작 =====
읽기 성공 | 5월 통합 (0010_cnt, 60plus_cnt, 4050_cnt제거, 출도착시간 필터).csv | 인코딩: utf-8-sig | 행: 26,832,437
중복 제거 행 수: 0

===== 6월 처리 시작 =====
읽기 성공 | 6월 통합 (0010_cnt, 60plus_cnt, 4050_cnt제거, 출도착시간 필터).csv | 인코딩: utf-8-sig | 행: 25,963,477
중복 제거 행 수: 0

===== 7월 처리 시작 =====
읽기 성공 | 7월 통합 (0010_cnt, 60plus_cnt, 4050_cnt제거, 출도착시간 필터).

In [130]:
youth_mobility = pd.concat(
    monthly_data_list,
    ignore_index=True
)


print("병합 후 데이터 크기:", youth_mobility.shape)
print("기준월 목록:", sorted(youth_mobility["기준월"].unique()))

youth_mobility.head()

병합 후 데이터 크기: (318974147, 10)
기준월 목록: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]


,o_admdong_cd,d_admdong_cd,st_time_cd,fns_time_cd,move_dist,move_time,2030_cnt,total_cnt,etl_ymd,기준월
0,11110515,11110515,700,700,610,3,23.90,68.48,2025-01-01,1
1,11110515,11110530,700,700,907,7,5.08,5.08,2025-01-01,1
2,11110515,11110540,700,700,1600,2,0.00,2.75,2025-01-01,1
3,11110515,11110570,700,700,939,7,2.07,2.07,2025-01-01,1
4,11110515,11110580,700,700,1461,6,5.03,7.73,2025-01-01,1


In [131]:
print("전체 행 수:", f"{len(youth_mobility):,}")
print("전체 열 수:", len(youth_mobility.columns))

print(
    "날짜 변환 실패:",
    youth_mobility["etl_ymd"].isna().sum()
)

print(
    "출발 행정동 코드 결측:",
    youth_mobility["o_admdong_cd"].isna().sum()
)

print(
    "도착 행정동 코드 결측:",
    youth_mobility["d_admdong_cd"].isna().sum()
)

전체 행 수: 318,974,147
전체 열 수: 10
날짜 변환 실패: 0
출발 행정동 코드 결측: 0
도착 행정동 코드 결측: 0


In [132]:
before_filter_count = len(youth_mobility)


youth_mobility_seoul = youth_mobility[
    youth_mobility["o_admdong_cd"].str.startswith(
        "11",
        na=False
    )
    &
    youth_mobility["d_admdong_cd"].str.startswith(
        "11",
        na=False
    )
].copy()


after_filter_count = len(youth_mobility_seoul)


print("필터링 전 행 수:", f"{before_filter_count:,}")
print("서울→서울 행 수:", f"{after_filter_count:,}")
print(
    "서울 외 이동 제외 행 수:",
    f"{before_filter_count - after_filter_count:,}"
)

필터링 전 행 수: 318,974,147
서울→서울 행 수: 217,856,493
서울 외 이동 제외 행 수: 101,117,654


In [133]:
print(
    "서울 외 출발 코드 수:",
    (
        ~youth_mobility_seoul["o_admdong_cd"]
        .str.startswith("11", na=False)
    ).sum()
)

print(
    "서울 외 도착 코드 수:",
    (
        ~youth_mobility_seoul["d_admdong_cd"]
        .str.startswith("11", na=False)
    ).sum()
)

서울 외 출발 코드 수: 0
서울 외 도착 코드 수: 0


In [152]:
# =========================================================
# 구·신 행정동 코드표 통합
# =========================================================

ADM_OLD_FILE = RAW_DIR / "seoul_admi_list_2023_to_202507.csv"
ADM_NEW_FILE = RAW_DIR / "seoul_admi_list_202508_to_202606.csv"


# 2025년 7월까지 사용된 행정동 코드표
admdong_ref_old = pd.read_csv(
    ADM_OLD_FILE,
    encoding="utf-8-sig",
    dtype={
        "자치구": "string",
        "행정동코드": "string",
        "행정동명": "string"
    }
)


# 2025년 8월 이후 사용된 행정동 코드표
admdong_ref_new = pd.read_csv(
    ADM_NEW_FILE,
    encoding="utf-8-sig",
    dtype={
        "자치구": "string",
        "행정동코드": "string",
        "행정동명": "string"
    }
)


# 어느 코드표에서 가져왔는지 기록
admdong_ref_old["코드기준"] = "2023~2025.07"
admdong_ref_new["코드기준"] = "2025.08~2026.06"


# 두 코드표 통합
admdong_ref = pd.concat(
    [
        admdong_ref_old,
        admdong_ref_new
    ],
    ignore_index=True
)


# 문자열 정리
for col in ["자치구", "행정동명"]:
    admdong_ref[col] = (
        admdong_ref[col]
        .astype("string")
        .str.strip()
    )


admdong_ref["행정동코드"] = (
    admdong_ref["행정동코드"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"[^0-9]", "", regex=True)
    .str.zfill(8)
)


# 동일한 코드가 양쪽 파일에 반복되는 경우 한 행만 유지
# 최신 코드표를 뒤에 붙였으므로 keep="last"로 최신 명칭 우선
admdong_ref = (
    admdong_ref
    .drop_duplicates(
        subset=["행정동코드"],
        keep="last"
    )
    .sort_values("행정동코드")
    .reset_index(drop=True)
)


print("구 코드표 크기:", admdong_ref_old.shape)
print("신 코드표 크기:", admdong_ref_new.shape)
print("통합 코드표 크기:", admdong_ref.shape)

print("\n통합 코드표 컬럼:")
print(admdong_ref.columns.tolist())


# 용신동 개편 관련 코드 확인
dongdaemun_check_codes = [
    "11230536",  # 구 용신동
    "11230515",  # 신설동
    "11230533"   # 용두동
]

display(
    admdong_ref[
        admdong_ref["행정동코드"].isin(
            dongdaemun_check_codes
        )
    ]
)

구 코드표 크기: (426, 4)
신 코드표 크기: (427, 4)
통합 코드표 크기: (428, 4)

통합 코드표 컬럼:
['자치구', '행정동코드', '행정동명', '코드기준']


,자치구,행정동코드,행정동명,코드기준
80,동대문구,11230515,신설동,2025.08~2026.06
81,동대문구,11230533,용두동,2025.08~2026.06
82,동대문구,11230536,용신동,2023~2025.07


In [153]:
# 병합용 출발 행정동 코드표
origin_ref = admdong_ref[
    [
        "행정동코드",
        "자치구",
        "행정동명"
    ]
].rename(
    columns={
        "행정동코드": "o_admdong_cd",
        "자치구": "o_gu_name",
        "행정동명": "o_admdong_name"
    }
)


# 병합용 도착 행정동 코드표
destination_ref = admdong_ref[
    [
        "행정동코드",
        "자치구",
        "행정동명"
    ]
].rename(
    columns={
        "행정동코드": "d_admdong_cd",
        "자치구": "d_gu_name",
        "행정동명": "d_admdong_name"
    }
)


# 출발 행정동명 병합
youth_mobility_seoul_named = youth_mobility_seoul.merge(
    origin_ref,
    on="o_admdong_cd",
    how="left",
    validate="many_to_one"
)


# 도착 행정동명 병합
youth_mobility_seoul_named = youth_mobility_seoul_named.merge(
    destination_ref,
    on="d_admdong_cd",
    how="left",
    validate="many_to_one"
)

In [154]:
mapping_check = pd.Series({
    "전체 행 수": len(youth_mobility_seoul_named),

    "출발 자치구명 누락":
        youth_mobility_seoul_named[
            "o_gu_name"
        ].isna().sum(),

    "출발 행정동명 누락":
        youth_mobility_seoul_named[
            "o_admdong_name"
        ].isna().sum(),

    "도착 자치구명 누락":
        youth_mobility_seoul_named[
            "d_gu_name"
        ].isna().sum(),

    "도착 행정동명 누락":
        youth_mobility_seoul_named[
            "d_admdong_name"
        ].isna().sum(),

    "완전 중복 행 수":
        youth_mobility_seoul_named
        .duplicated()
        .sum()
})

mapping_check

전체 행 수        217856493
출발 자치구명 누락            0
출발 행정동명 누락            0
도착 자치구명 누락            0
도착 행정동명 누락            0
완전 중복 행 수             0
dtype: int64

In [155]:
youth_mobility_seoul_named = (
    youth_mobility_seoul
    .merge(
        origin_ref,
        on="o_admdong_cd",
        how="left",
        validate="many_to_one"
    )
    .merge(
        destination_ref,
        on="d_admdong_cd",
        how="left",
        validate="many_to_one"
    )
)


print("매핑 전 행 수:", f"{len(youth_mobility_seoul):,}")
print("매핑 후 행 수:", f"{len(youth_mobility_seoul_named):,}")

매핑 전 행 수: 217,856,493
매핑 후 행 수: 217,856,493


In [156]:
front_columns = [
    "기준월",
    "etl_ymd",

    "o_admdong_cd",
    "o_gu_name",
    "o_admdong_name",

    "d_admdong_cd",
    "d_gu_name",
    "d_admdong_name",

    "st_time_cd",
    "fns_time_cd",
    "move_dist",
    "move_time",
    "2030_cnt",
    "total_cnt"
]


youth_mobility_seoul_named = (
    youth_mobility_seoul_named[
        front_columns
    ]
    .sort_values(
        [
            "etl_ymd",
            "o_admdong_cd",
            "d_admdong_cd",
            "st_time_cd"
        ]
    )
    .reset_index(drop=True)
)


youth_mobility_seoul_named.head()

,기준월,etl_ymd,o_admdong_cd,o_gu_name,o_admdong_name,d_admdong_cd,d_gu_name,d_admdong_name,st_time_cd,fns_time_cd,move_dist,move_time,2030_cnt,total_cnt
0,1,2025-01-01,11110515,종로구,청운효자동,11110515,종로구,청운효자동,700,700,610,3,23.90,68.48
1,1,2025-01-01,11110515,종로구,청운효자동,11110515,종로구,청운효자동,700,720,695,10,5.00,31.06
2,1,2025-01-01,11110515,종로구,청운효자동,11110515,종로구,청운효자동,700,740,308,44,0.00,8.51
3,1,2025-01-01,11110515,종로구,청운효자동,11110515,종로구,청운효자동,700,820,1323,76,2.48,2.48
4,1,2025-01-01,11110515,종로구,청운효자동,11110515,종로구,청운효자동,700,840,390,105,0.00,2.60


In [157]:
mapping_check = pd.Series({
    "전체 행 수":
        len(youth_mobility_seoul_named),

    "출발 자치구명 누락":
        youth_mobility_seoul_named[
            "o_gu_name"
        ].isna().sum(),

    "출발 행정동명 누락":
        youth_mobility_seoul_named[
            "o_admdong_name"
        ].isna().sum(),

    "도착 자치구명 누락":
        youth_mobility_seoul_named[
            "d_gu_name"
        ].isna().sum(),

    "도착 행정동명 누락":
        youth_mobility_seoul_named[
            "d_admdong_name"
        ].isna().sum(),

    "완전 중복 행 수":
        youth_mobility_seoul_named
        .duplicated()
        .sum()
})


mapping_check

전체 행 수        217856493
출발 자치구명 누락            0
출발 행정동명 누락            0
도착 자치구명 누락            0
도착 행정동명 누락            0
완전 중복 행 수             0
dtype: int64

In [158]:
unmapped_origin_codes = (
    youth_mobility_seoul_named.loc[
        youth_mobility_seoul_named[
            "o_admdong_name"
        ].isna(),
        "o_admdong_cd"
    ]
    .drop_duplicates()
    .sort_values()
)


unmapped_destination_codes = (
    youth_mobility_seoul_named.loc[
        youth_mobility_seoul_named[
            "d_admdong_name"
        ].isna(),
        "d_admdong_cd"
    ]
    .drop_duplicates()
    .sort_values()
)


print("매핑되지 않은 출발 코드")
print(unmapped_origin_codes.tolist())

print("\n매핑되지 않은 도착 코드")
print(unmapped_destination_codes.tolist())

매핑되지 않은 출발 코드
[]

매핑되지 않은 도착 코드
[]


# 날짜,시간 형식 정리

In [159]:
# =========================================================
# 저장 전 날짜·시간 형식 정리
# =========================================================

# 1. 날짜 변수 날짜형으로 변환
youth_mobility_seoul_named["etl_ymd"] = pd.to_datetime(
    youth_mobility_seoul_named["etl_ymd"],
    errors="coerce"
)


# 날짜 변환 실패 확인
date_error_count = (
    youth_mobility_seoul_named["etl_ymd"]
    .isna()
    .sum()
)

print("날짜 변환 실패 행 수:", f"{date_error_count:,}")


if date_error_count > 0:
    raise ValueError(
        "etl_ymd 날짜 변환에 실패한 행이 있습니다."
    )


# 2. 700 → 07:00, 1900 → 19:00 형식으로 변환하는 함수
def format_time_code(series: pd.Series) -> pd.Series:
    """
    숫자형 시간 코드를 HH:MM 문자열로 변환한다.

    예:
    700  -> 07:00
    720  -> 07:20
    1900 -> 19:00
    """

    numeric_time = pd.to_numeric(
        series,
        errors="coerce"
    ).astype("Int64")


    hour = numeric_time // 100
    minute = numeric_time % 100


    # 정상적인 시간 범위인지 확인
    invalid_mask = (
        numeric_time.notna()
        &
        (
            hour.lt(0)
            | hour.gt(23)
            | minute.lt(0)
            | minute.gt(59)
        )
    )


    if invalid_mask.any():
        invalid_values = (
            numeric_time[invalid_mask]
            .drop_duplicates()
            .sort_values()
            .tolist()
        )

        raise ValueError(
            "올바르지 않은 시간 코드가 있습니다: "
            f"{invalid_values[:20]}"
        )


    formatted_time = (
        hour.astype("string").str.zfill(2)
        + ":"
        + minute.astype("string").str.zfill(2)
    )


    # 원래 결측값은 결측값으로 유지
    formatted_time = formatted_time.mask(
        numeric_time.isna(),
        pd.NA
    )


    return formatted_time


# 원본 시간 코드는 보존
youth_mobility_seoul_named["st_time_cd_original"] = (
    youth_mobility_seoul_named["st_time_cd"]
)

youth_mobility_seoul_named["fns_time_cd_original"] = (
    youth_mobility_seoul_named["fns_time_cd"]
)


# 시간 표시용 컬럼 변환
youth_mobility_seoul_named["st_time_cd"] = format_time_code(
    youth_mobility_seoul_named["st_time_cd"]
)

youth_mobility_seoul_named["fns_time_cd"] = format_time_code(
    youth_mobility_seoul_named["fns_time_cd"]
)


print("날짜 dtype:", youth_mobility_seoul_named["etl_ymd"].dtype)

print(
    youth_mobility_seoul_named[
        [
            "etl_ymd",
            "st_time_cd_original",
            "st_time_cd",
            "fns_time_cd_original",
            "fns_time_cd"
        ]
    ].head(10)
)

날짜 변환 실패 행 수: 0
날짜 dtype: datetime64[us]
     etl_ymd  st_time_cd_original st_time_cd  fns_time_cd_original fns_time_cd
0 2025-01-01                  700      07:00                   700       07:00
1 2025-01-01                  700      07:00                   720       07:20
2 2025-01-01                  700      07:00                   740       07:40
3 2025-01-01                  700      07:00                   820       08:20
4 2025-01-01                  700      07:00                   840       08:40
5 2025-01-01                  700      07:00                   900       09:00
6 2025-01-01                  720      07:20                   720       07:20
7 2025-01-01                  720      07:20                   740       07:40
8 2025-01-01                  740      07:40                   740       07:40
9 2025-01-01                  740      07:40                   800       08:00


In [160]:
OUTPUT_FILE = (
    PROCESSED_DIR
    / "youth_mobility_seoul_cleaned.csv"
)


youth_mobility_seoul_named.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d"
)


print("저장 완료:", OUTPUT_FILE)
print("파일 존재:", OUTPUT_FILE.exists())

print(
    "파일 크기:",
    f"{OUTPUT_FILE.stat().st_size / 1024 / 1024:.2f} MB"
)

저장 완료: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/youth_mobility_seoul_cleaned.csv
파일 존재: True
파일 크기: 23257.55 MB


In [161]:
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 1. 저장된 데이터 다시 불러오기
# ============================================================

check_df = pd.read_csv(
    OUTPUT_FILE,
    encoding="utf-8-sig",
    low_memory=False
)

print("데이터 불러오기 완료")
print("데이터 크기:", check_df.shape)
print("전체 행 수:", len(check_df))
print("전체 변수 수:", len(check_df.columns))


# ============================================================
# 2. 변수명 확인
# ============================================================

print("\n" + "=" * 70)
print("1. 전체 변수명")
print("=" * 70)

for number, column in enumerate(check_df.columns, start=1):
    print(f"{number:>2}. {column}")


# 중복된 변수명이 있는지 확인
duplicated_columns = check_df.columns[
    check_df.columns.duplicated()
].tolist()

print("\n중복 변수명:", duplicated_columns)

if len(duplicated_columns) == 0:
    print("중복된 변수명 없음")


# 변수명 앞뒤 공백 확인
space_columns = [
    column
    for column in check_df.columns
    if column != column.strip()
]

print("앞뒤 공백이 있는 변수명:", space_columns)

if len(space_columns) == 0:
    print("앞뒤 공백이 있는 변수명 없음")


# ============================================================
# 3. 변수별 자료형 및 고유값 개수
# ============================================================

print("\n" + "=" * 70)
print("2. 변수별 자료형과 고유값 개수")
print("=" * 70)

variable_summary = pd.DataFrame({
    "변수명": check_df.columns,
    "자료형": check_df.dtypes.astype(str).values,
    "고유값수": [
        check_df[column].nunique(dropna=True)
        for column in check_df.columns
    ],
    "결측치수": check_df.isna().sum().values,
    "결측치비율(%)": (
        check_df.isna().mean().values * 100
    ).round(2)
})

display(variable_summary)


# ============================================================
# 4. 수치형·범주형 변수 구분
# ============================================================

numeric_columns = check_df.select_dtypes(
    include="number"
).columns.tolist()

categorical_columns = check_df.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("\n" + "=" * 70)
print("3. 변수 유형 구분")
print("=" * 70)

print("수치형 변수 수:", len(numeric_columns))
print(numeric_columns)

print("\n범주형·문자형 변수 수:", len(categorical_columns))
print(categorical_columns)


# ============================================================
# 5. 수치형 변수 기초통계
# ============================================================

print("\n" + "=" * 70)
print("4. 수치형 변수 기초통계")
print("=" * 70)

if len(numeric_columns) > 0:

    numeric_summary = (
        check_df[numeric_columns]
        .describe(
            percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
        )
        .T
    )

    # 중앙값을 별도 이름으로 확인하기 쉽게 추가
    numeric_summary["median"] = (
        check_df[numeric_columns].median()
    )

    # 표준편차가 평균 대비 어느 정도인지 확인
    numeric_summary["변동계수_CV"] = (
        numeric_summary["std"]
        / numeric_summary["mean"].abs()
    ).round(4)

    display(numeric_summary)

else:
    print("수치형 변수가 없습니다.")


# ============================================================
# 6. 범주형 변수 기초통계
# ============================================================

print("\n" + "=" * 70)
print("5. 범주형 변수 기초통계")
print("=" * 70)

if len(categorical_columns) > 0:

    categorical_summary = (
        check_df[categorical_columns]
        .describe()
        .T
    )

    display(categorical_summary)

else:
    print("범주형 변수가 없습니다.")


# ============================================================
# 7. 결측치 상세 확인
# ============================================================

print("\n" + "=" * 70)
print("6. 결측치 확인")
print("=" * 70)

missing_summary = pd.DataFrame({
    "변수명": check_df.columns,
    "결측치수": check_df.isna().sum().values,
    "결측치비율(%)": (
        check_df.isna().mean().values * 100
    ).round(2)
})

missing_summary = (
    missing_summary[
        missing_summary["결측치수"] > 0
    ]
    .sort_values(
        "결측치비율(%)",
        ascending=False
    )
    .reset_index(drop=True)
)

if missing_summary.empty:
    print("결측치가 없습니다.")

else:
    display(missing_summary)

    print(
        "결측치가 하나라도 있는 행 수:",
        check_df.isna().any(axis=1).sum()
    )

    print(
        "모든 값이 결측치인 행 수:",
        check_df.isna().all(axis=1).sum()
    )


# ============================================================
# 8. 빈 문자열 및 공백 문자열 확인
# ============================================================

print("\n" + "=" * 70)
print("7. 빈 문자열 확인")
print("=" * 70)

empty_string_result = []

for column in categorical_columns:

    string_values = (
        check_df[column]
        .dropna()
        .astype(str)
        .str.strip()
    )

    empty_count = string_values.eq("").sum()

    if empty_count > 0:
        empty_string_result.append({
            "변수명": column,
            "빈문자열수": empty_count,
            "빈문자열비율(%)": round(
                empty_count / len(check_df) * 100,
                2
            )
        })

empty_string_summary = pd.DataFrame(empty_string_result)

if empty_string_summary.empty:
    print("빈 문자열 또는 공백 문자열이 없습니다.")

else:
    display(empty_string_summary)


# ============================================================
# 9. 완전 중복 행 확인
# ============================================================

print("\n" + "=" * 70)
print("8. 중복 행 확인")
print("=" * 70)

duplicate_count = check_df.duplicated().sum()

print("완전히 동일한 중복 행 수:", duplicate_count)
print(
    "중복 행 비율:",
    f"{duplicate_count / len(check_df) * 100:.2f}%"
)

if duplicate_count > 0:

    duplicate_rows = check_df[
        check_df.duplicated(keep=False)
    ].sort_values(
        by=check_df.columns.tolist()
    )

    display(duplicate_rows.head(20))


# ============================================================
# 10. 범주형 변수별 값 분포
# ============================================================

print("\n" + "=" * 70)
print("9. 범주형 변수별 값 분포")
print("=" * 70)

for column in categorical_columns:

    unique_count = check_df[column].nunique(dropna=False)

    print("\n" + "-" * 60)
    print(f"변수명: {column}")
    print(f"고유값 수: {unique_count}")

    value_distribution = (
        check_df[column]
        .fillna("[결측치]")
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="빈도")
    )

    value_distribution["비율(%)"] = (
        value_distribution["빈도"]
        / len(check_df)
        * 100
    ).round(2)

    # 값이 너무 많을 수 있으므로 상위 20개만 출력
    display(value_distribution.head(20))


# ============================================================
# 11. 수치형 변수 분포 시각화
# ============================================================

print("\n" + "=" * 70)
print("10. 수치형 변수 분포")
print("=" * 70)

for column in numeric_columns:

    valid_data = check_df[column].dropna()

    if valid_data.empty:
        print(f"{column}: 유효한 값이 없어 생략")
        continue

    print(
        f"{column}: "
        f"최솟값={valid_data.min()}, "
        f"중앙값={valid_data.median()}, "
        f"평균={valid_data.mean():.2f}, "
        f"최댓값={valid_data.max()}"
    )

    plt.figure(figsize=(8, 4))

    plt.hist(
        valid_data,
        bins=30,
        edgecolor="black"
    )

    plt.title(f"{column} 분포")
    plt.xlabel(column)
    plt.ylabel("빈도")
    plt.tight_layout()
    plt.show()


# ============================================================
# 12. 수치형 변수 이상치 개수 확인
#     IQR 기준
# ============================================================

print("\n" + "=" * 70)
print("11. 수치형 변수 이상치 확인")
print("=" * 70)

outlier_result = []

for column in numeric_columns:

    valid_data = check_df[column].dropna()

    if valid_data.empty:
        continue

    q1 = valid_data.quantile(0.25)
    q3 = valid_data.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_count = (
        (valid_data < lower_bound)
        | (valid_data > upper_bound)
    ).sum()

    outlier_result.append({
        "변수명": column,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "하한값": lower_bound,
        "상한값": upper_bound,
        "이상치수": outlier_count,
        "이상치비율(%)": round(
            outlier_count / len(valid_data) * 100,
            2
        )
    })

outlier_summary = pd.DataFrame(outlier_result)

if outlier_summary.empty:
    print("확인할 수치형 변수가 없습니다.")

else:
    outlier_summary = outlier_summary.sort_values(
        "이상치비율(%)",
        ascending=False
    )

    display(outlier_summary)


# ============================================================
# 13. 최종 데이터 샘플 확인
# ============================================================

print("\n" + "=" * 70)
print("12. 데이터 샘플")
print("=" * 70)

display(check_df.head(10))

Python(4306) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(4307) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Matplotlib is building the font cache; this may take a moment.


: 